### Notebook 01 — Bigram Language Model

#### Introdução

Neste notebook construiremos nosso primeiro modelo de linguagem.

Embora extremamente simples, ele contém várias das ideias fundamentais presentes em sistemas modernos de IA, incluindo os Large Language Models (LLMs).

O objetivo não é criar um modelo poderoso, mas compreender os princípios básicos que permitiram a evolução da área.

Ao longo desta trilha veremos a seguinte progressão:

Bigram
→ Markov
→ HMM
→ Redes Neurais
→ Embeddings
→ Attention
→ Transformers
→ LLMs

Este notebook representa o primeiro passo dessa jornada.

---

#### O que vamos aprender

Ao final deste notebook você deverá entender:

- O que é um token
- O que é um vocabulário
- Como texto vira números
- O que é uma probabilidade condicional
- Como um modelo gera texto
- O que é uma Cadeia de Markov
- Por que contexto limitado é um problema

### 1. Carregando o Dataset

#### Objetivo

Todo modelo de linguagem precisa aprender a partir de exemplos.

Esses exemplos compõem o **dataset**, que nada mais é do que um conjunto de textos utilizados durante o treinamento.

Neste primeiro notebook utilizaremos um dataset extremamente pequeno, pois nosso objetivo não é construir um modelo poderoso, mas entender os conceitos fundamentais por trás dos modelos de linguagem.

Mais adiante utilizaremos datasets maiores e mais realistas.

---

#### O que é um dataset?

Um dataset é o conjunto de informações utilizado para ensinar o modelo.

Exemplo:

- livros
- artigos
- diálogos
- código-fonte
- páginas da internet

Modelos modernos são treinados com bilhões ou até trilhões de tokens.

Nosso primeiro modelo será treinado com apenas algumas linhas de texto.

In [31]:
# ==================================================
# SEÇÃO 1 - CARREGAR DATASET
# ==================================================

with open("../data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text)

ola mundo
o gato dormiu
o cachorro latiu
ola chatgpt


#### Exercício de Reflexão

Observe o conteúdo carregado.

Pergunta:

O modelo está enxergando palavras, frases ou apenas texto bruto?

Pense na resposta antes de continuar.

#### Conceito Importante

Neste momento ainda não existe tokenização.

O modelo possui apenas uma sequência de caracteres.

Para o computador, texto é apenas uma sequência de símbolos armazenados na memória.

---
### 2. Criando o Vocabulário

#### Objetivo

Descobrir todos os símbolos únicos presentes no dataset.

Esse conjunto de símbolos é chamado de **vocabulário**.

Todo modelo de linguagem possui algum tipo de vocabulário.

Sem ele, o modelo não consegue representar texto matematicamente.

---

#### O que é um vocabulário?

Vocabulário é o conjunto de tokens que o modelo conhece.

Neste notebook cada caractere será tratado como um token.

Exemplo:

Texto:

"ola"

Vocabulário:

['o', 'l', 'a']

Quantidade de tokens:

3

In [32]:
# ==================================================
# SEÇÃO 2 - VOCABULÁRIO
# ==================================================

chars = sorted(list(set(text)))

vocab_size = len(chars)

print(chars)
print()
print(f"Vocabulary Size: {vocab_size}")

['\n', ' ', 'a', 'c', 'd', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 't', 'u']

Vocabulary Size: 16


#### Exercício de Reflexão

Se adicionarmos um novo caractere ao dataset, por exemplo:

@

O tamanho do vocabulário aumenta ou permanece igual?

Por quê?

#### Conceito Importante

O vocabulário define todos os símbolos que o modelo é capaz de representar.

Se um símbolo não estiver no vocabulário, o modelo não saberá como processá-lo.

---
### 3. Tokenização

#### Objetivo

Converter texto em números.

Modelos de linguagem não trabalham diretamente com texto.

Toda informação precisa ser representada numericamente antes de ser processada.

Esse processo é chamado de **tokenização**.

#### Exemplo Visual

Texto:

ola

↓

Tokens:

['o', 'l', 'a']

↓

IDs:

[12, 8, 3]

---

#### Por que converter texto para números?

Operações matemáticas são realizadas sobre números.

Como redes neurais trabalham com álgebra linear, precisamos transformar cada token em uma representação numérica.

In [33]:
# ==================================================
# SEÇÃO 3 - TOKENIZAÇÃO
# ==================================================

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

print("Encode:")
print(encode("ola"))

print()

print("Decode:")
print(decode(encode("ola")))

Encode:
[11, 8, 2]

Decode:
ola


#### Exercício de Reflexão

Considere o texto:

"gato"

Pergunta:

O modelo está armazenando a palavra "gato" ou apenas os IDs correspondentes aos caracteres?

O que acontece com o significado da palavra durante essa transformação?

#### Conceito Importante

Tokenização não significa compreensão.

O modelo apenas substitui símbolos por números.

O significado ainda não existe para a máquina.

Ele será aprendido posteriormente através das relações estatísticas observadas durante o treinamento.

---
### 4. Transformando o Dataset em Tokens

#### Objetivo

Aplicar o processo de tokenização ao dataset inteiro.

A partir deste momento o modelo passará a trabalhar apenas com números.

Esse é exatamente o mesmo princípio utilizado em modelos modernos, embora os tokens sejam muito mais sofisticados.

In [34]:
# ==================================================
# SEÇÃO 4 - DATASET TOKENIZADO
# ==================================================

data = encode(text)

print(data[:100])

[11, 8, 2, 1, 9, 15, 10, 4, 11, 0, 11, 1, 5, 2, 14, 11, 1, 4, 11, 13, 9, 7, 15, 0, 11, 1, 3, 2, 3, 6, 11, 13, 13, 11, 1, 8, 2, 14, 7, 15, 0, 11, 8, 2, 1, 3, 6, 2, 14, 5, 12, 14]


#### Exercício de Reflexão

Observe a saída gerada.

Você ainda consegue identificar facilmente o texto original?

Por que essa representação numérica é mais útil para um modelo computacional do que texto puro?

#### Conceito Importante

A partir daqui o modelo não enxerga mais palavras nem frases.

Ele vê apenas uma sequência de IDs.

Todo o processamento realizado por modelos de linguagem acontece sobre essas representações numéricas.

#### Recapitulando

Até agora construímos o seguinte pipeline:

Texto
→ Vocabulário
→ Tokenização
→ Sequência Numérica

Ainda não existe aprendizado.

Na próxima seção o modelo começará a identificar padrões estatísticos.

---
### 5. Construindo os Bigramas

#### Objetivo

Agora vamos começar a extrair conhecimento do dataset.

Até este ponto, nosso modelo apenas armazenava texto em formato numérico.

A partir desta seção ele começará a observar relações entre tokens consecutivos.

Essas relações são chamadas de **bigramas**.

---

#### O que é um Bigrama?

Um bigrama é um par formado por dois tokens consecutivos.

Exemplo:

Texto:

"ola"

Tokens:

o → l → a

Bigramas:

(o, l)

(l, a)

---

#### O que o modelo está tentando descobrir?

A pergunta central é:

"Dado o token atual, qual token costuma aparecer em seguida?"

Essa é uma das ideias fundamentais dos modelos de linguagem.

Mesmo modelos modernos como GPT continuam tentando prever o próximo token.

A diferença está na quantidade de contexto utilizada para fazer essa previsão.

In [35]:
# ==================================================
# SEÇÃO 5 - CONSTRUÇÃO DOS BIGRAMAS
# ==================================================
#
# Vamos percorrer a sequência de tokens e contar
# quantas vezes cada transição ocorre.
#
# Exemplo:
#
# o -> l
# o -> espaço
# l -> a
#
# ==================================================

bigrams = {}

for i in range(len(data) - 1):

    current_token = data[i]
    next_token = data[i + 1]

    if current_token not in bigrams:
        bigrams[current_token] = {}

    if next_token not in bigrams[current_token]:
        bigrams[current_token][next_token] = 0

    bigrams[current_token][next_token] += 1

In [36]:
# ==================================================
# VISUALIZANDO TRANSIÇÕES
# ==================================================

for current_token in bigrams:

    current_char = itos[current_token]

    print(f"\nDepois de '{current_char}'")

    for next_token, count in bigrams[current_token].items():

        next_char = itos[next_token]

        print(
            f"  -> '{next_char}' : {count}"
        )


Depois de 'o'
  -> 'l' : 2
  -> '
' : 1
  -> ' ' : 4
  -> 'r' : 2

Depois de 'l'
  -> 'a' : 3

Depois de 'a'
  -> ' ' : 2
  -> 't' : 3
  -> 'c' : 1

Depois de ' '
  -> 'm' : 1
  -> 'g' : 1
  -> 'd' : 1
  -> 'c' : 2
  -> 'l' : 1

Depois de 'm'
  -> 'u' : 1
  -> 'i' : 1

Depois de 'u'
  -> 'n' : 1
  -> '
' : 2

Depois de 'n'
  -> 'd' : 1

Depois de 'd'
  -> 'o' : 2

Depois de '
'
  -> 'o' : 3

Depois de 'g'
  -> 'a' : 1
  -> 'p' : 1

Depois de 't'
  -> 'o' : 1
  -> 'i' : 1
  -> 'g' : 1

Depois de 'r'
  -> 'm' : 1
  -> 'r' : 1
  -> 'o' : 1

Depois de 'i'
  -> 'u' : 2

Depois de 'c'
  -> 'a' : 1
  -> 'h' : 2

Depois de 'h'
  -> 'o' : 1
  -> 'a' : 1

Depois de 'p'
  -> 't' : 1


#### Exercício de Reflexão

Observe as transições encontradas.

Pergunta:

Se o caractere "o" aparece várias vezes no dataset, ele sempre será seguido pelo mesmo caractere?

O que isso nos diz sobre a natureza da linguagem?

Pense nisso antes de continuar.

#### Conceito Importante

Neste momento o modelo ainda não entende significado.

Ele não sabe o que é:

- cachorro
- gato
- dormir
- correr

Ele apenas observa padrões estatísticos.

O conhecimento do modelo é composto exclusivamente pelas frequências observadas durante o treinamento.

Essa característica também está presente nos modelos modernos.

A diferença é que eles conseguem capturar padrões muito mais complexos.

---

### 6. Transformando Contagens em Probabilidades

#### Objetivo

Até agora nosso modelo aprendeu apenas frequências.

Exemplo:

Depois do token "o":

- "l" apareceu 5 vezes
- espaço apareceu 3 vezes

Mas para fazer previsões precisamos transformar essas frequências em probabilidades.

---

#### Por que probabilidades?

Imagine que queremos prever o próximo token após "o".

Se utilizarmos apenas as contagens:

- l → 5
- espaço → 3

Ainda não sabemos qual a chance real de cada opção.

Precisamos normalizar esses valores.

---

#### Intuição

Se um evento ocorreu:

- 5 vezes em um total de 8

Então sua probabilidade é:

5 / 8

Ou:

62.5%

É exatamente isso que faremos nesta seção.

In [37]:
# ==================================================
# SEÇÃO 6 - PROBABILIDADES
# ==================================================
#
# Converter contagens em probabilidades.
#
# Exemplo:
#
# o -> l : 5
# o -> espaço : 3
#
# total = 8
#
# o -> l : 5/8 = 0.625
# o -> espaço : 3/8 = 0.375
#
# ==================================================

probabilities = {}

for current_token, transitions in bigrams.items():

    total = sum(transitions.values())

    probabilities[current_token] = {}

    for next_token, count in transitions.items():

        probabilities[current_token][next_token] = (
            count / total
        )

In [38]:
# ==================================================
# VISUALIZANDO PROBABILIDADES
# ==================================================

for current_token in probabilities:

    current_char = itos[current_token]

    print(f"\nDepois de '{current_char}'")

    for next_token, prob in probabilities[current_token].items():

        next_char = itos[next_token]

        print(
            f"  -> '{next_char}' : {prob:.3f}"
        )


Depois de 'o'
  -> 'l' : 0.222
  -> '
' : 0.111
  -> ' ' : 0.444
  -> 'r' : 0.222

Depois de 'l'
  -> 'a' : 1.000

Depois de 'a'
  -> ' ' : 0.333
  -> 't' : 0.500
  -> 'c' : 0.167

Depois de ' '
  -> 'm' : 0.167
  -> 'g' : 0.167
  -> 'd' : 0.167
  -> 'c' : 0.333
  -> 'l' : 0.167

Depois de 'm'
  -> 'u' : 0.500
  -> 'i' : 0.500

Depois de 'u'
  -> 'n' : 0.333
  -> '
' : 0.667

Depois de 'n'
  -> 'd' : 1.000

Depois de 'd'
  -> 'o' : 1.000

Depois de '
'
  -> 'o' : 1.000

Depois de 'g'
  -> 'a' : 0.500
  -> 'p' : 0.500

Depois de 't'
  -> 'o' : 0.333
  -> 'i' : 0.333
  -> 'g' : 0.333

Depois de 'r'
  -> 'm' : 0.333
  -> 'r' : 0.333
  -> 'o' : 0.333

Depois de 'i'
  -> 'u' : 1.000

Depois de 'c'
  -> 'a' : 0.333
  -> 'h' : 0.667

Depois de 'h'
  -> 'o' : 0.500
  -> 'a' : 0.500

Depois de 'p'
  -> 't' : 1.000


#### Exercício de Reflexão

Observe as probabilidades calculadas.

Pergunta:

Se após o token "o" encontramos:

- l → 0.75
- espaço → 0.25

Isso significa que o modelo SEMPRE escolherá "l"?

Ou apenas que "l" é mais provável?

Qual seria a vantagem de permitir escolhas diferentes ocasionalmente?

#### Conceito Importante

Probabilidade não é certeza.

Uma probabilidade de 80% não significa que um evento acontecerá sempre.

Significa apenas que ele tende a acontecer com maior frequência.

Essa distinção é extremamente importante para entender modelos de linguagem.

Modelos modernos não escolhem necessariamente a opção mais provável.

Eles frequentemente realizam amostragem probabilística para gerar resultados mais variados e criativos.

---
### 7. A Matemática por Trás do Bigrama

#### Objetivo

Formalizar matematicamente o que acabamos de construir.

Nosso modelo responde sempre à mesma pergunta:

> Dado o token atual, qual é a probabilidade do próximo token?

Essa relação é chamada de probabilidade condicional.

---

#### Fórmula

$$
P(x_t \mid x_{t-1})
$$

Onde:

- $x_t$ representa o token atual
- $x_{t-1}$ representa o token anterior

Lemos essa expressão como:

> Probabilidade do token atual dado o token anterior.

---

#### Exemplo

Se observarmos a sequência:

```
o -> l -> a
```

O modelo pode aprender:

$$
P(l \mid o) = 0.75
$$

e

$$
P(\text{espaço} \mid o) = 0.25
$$

Isso significa que, após observar o token "o", existe 75% de chance de o próximo token ser "l".

#### Exercício de Reflexão

Considere as frases:

"O cachorro latiu."

"O cachorro correu."

Após a palavra "cachorro", existem múltiplas possibilidades.

Como um modelo baseado em bigramas decide qual delas escolher?

Quais limitações podem surgir quando observamos apenas um único token anterior?

#### Conceito Importante

A principal limitação do modelo de bigramas é sua memória extremamente curta.

Ele só consegue enxergar o token imediatamente anterior.

Todo o restante do contexto é descartado.

Essa limitação será uma das principais motivações para o surgimento dos próximos modelos que estudaremos.

---
### 8. Gerando Texto

#### Objetivo

Até agora nosso modelo aprendeu padrões estatísticos presentes no dataset.

Ele sabe, por exemplo:

- quais tokens costumam aparecer depois de outros
- com que frequência essas transições ocorrem
- qual a probabilidade de cada próxima escolha

Agora vamos utilizar esse conhecimento para gerar texto.

---

#### Como a geração funciona?

O processo é simples:

1. Escolhemos um token inicial.
2. Consultamos as probabilidades associadas a ele.
3. Sorteamos o próximo token.
4. O token escolhido torna-se o novo estado atual.
5. Repetimos o processo.

Essa estratégia é chamada de geração autoregressiva.

---

#### O que significa "autoregressivo"?

Significa que cada nova previsão depende das previsões anteriores.

O modelo gera um token por vez.

Depois utiliza esse token para gerar o próximo.

Esse mesmo princípio continua presente nos LLMs modernos.

In [44]:
# ==================================================
# SEÇÃO 8 - GERAÇÃO DE TEXTO
# ==================================================

import random

start_char = text[0]

current_token = stoi[start_char]

generated = start_char

for _ in range(200):

    if current_token not in probabilities:
        break

    next_tokens = list(
        probabilities[current_token].keys()
    )

    probs = list(
        probabilities[current_token].values()
    )

    next_token = random.choices(
        next_tokens,
        weights=probs,
        k=1
    )[0]

    generated += itos[next_token]

    current_token = next_token

print(generated)

olaca latgaca do catoro chato latormiu
orrola cho
o mundolatiundorrmiu
olatgptiu
ola gatgptgptolatgatiundo ga mu
o la do cholatgptiundolatgato miundoro
orolatiu
o gptgacho gatga miu
orrrmiundo latolato


#### Exercício de Reflexão

Execute a célula várias vezes.

Perguntas:

- O resultado é sempre igual?
- Quais partes tendem a se repetir?
- Quais partes mudam?

Por que isso acontece?

Observe que o modelo utiliza probabilidades, não regras fixas.

---
### 9. O Nascimento de um Modelo de Linguagem

Parabéns.

Você acabou de construir um modelo de linguagem funcional.

Ele é extremamente simples, mas já possui vários elementos fundamentais presentes em sistemas modernos:

- Tokenização
- Vocabulário
- Representação numérica
- Probabilidades
- Predição do próximo token
- Geração autoregressiva

---

#### O que é um Modelo de Linguagem?

Um modelo de linguagem é um sistema capaz de estimar probabilidades sobre sequências de tokens.

Em outras palavras, ele tenta responder perguntas como:

"Dado o contexto atual, qual token deve vir em seguida?"

Toda a evolução dos modelos de linguagem pode ser vista como uma tentativa de responder essa pergunta de forma cada vez mais precisa.

---

#### O que ele ainda não possui?

Nosso modelo ainda apresenta diversas limitações.

A principal delas é a memória.

Ele consegue enxergar apenas um token anterior.

Em outras palavras:

$$
P(x_t \mid x_{t-1})
$$

Todo o restante do contexto é descartado.

---

#### Exemplo

Considere as frases:

"O cachorro perseguiu o gato."

"O gato perseguiu o cachorro."

Nosso modelo observa apenas o token anterior.

Ele não compreende a estrutura completa da frase.

Isso limita drasticamente sua capacidade de representar significado.

---

#### O que vem depois?

A história da IA pode ser vista como uma sequência de tentativas de resolver esse problema.

Nos próximos notebooks veremos:

- Contextos maiores
- Cadeias de Markov
- Estados ocultos
- Redes neurais
- Embeddings
- Attention
- Transformers

Cada nova técnica surgirá para superar limitações da anterior.

---
### Resumo

Neste notebook aprendemos:

✅ O que é um token

✅ O que é um vocabulário

✅ Como texto vira números

✅ Como construir bigramas

✅ Como calcular probabilidades

✅ O que é uma probabilidade condicional

✅ Como gerar texto autoregressivamente

✅ Quais são as limitações dos modelos de bigrama

Agora estamos prontos para estudar modelos capazes de capturar mais contexto.

#### Desafio Opcional

Experimente:

- Alterar o dataset
- Adicionar novas frases
- Aumentar o tamanho do texto gerado
- Trocar o caractere inicial

Observe como essas mudanças afetam o comportamento do modelo.